Output: https://docs.google.com/spreadsheets/d/1UuwjLSfpOF-fYhOSv7Tw_iIJeqaAkItDZWURfo4cy1Q/edit?usp=sharing

## 1. Setup e imports

In [70]:
import re
import sys
import pandas as pd
from dotenv import load_dotenv

# Raiz del proyecto (ejecutar desde notebooks/)
sys.path.append("../")
load_dotenv("../.env")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

from src.config.settings import COUNTRY
from src.config.brand_configs import BRAND_FEES
from src.core.price_tracking.price_tracking import run_price_tracking
from src.utils.clean_model_name import clean_model_name
from src.utils.replace_model_name import map_model_name, load_mapping_file
from src.core.italika_pipeline import build_price_comparison
from src.sources.sheets.reader import GoogleSheetReader
from src.core.scraper.app import ScrapingUtils

gsheets = GoogleSheetReader()
scraper = ScrapingUtils()

## 2. Configuracion

In [30]:
BRAND_NAME = "Bajaj"

In [31]:
import json
from pathlib import Path

# Carga URLs desde el JSON correspondiente a la marca seleccionada
json_path = Path("../src/data/json/brand_urls") / f"{BRAND_NAME.lower()}.json"

with open(json_path, "r", encoding="utf-8") as f:
    urls_to_scrape = json.load(f)["urls"]

print(f"{len(urls_to_scrape)} URLs cargadas para {BRAND_NAME}")

24 URLs cargadas para Bajaj


## Lectura de base de inventario

Lectura desde archivo local

In [32]:
# CSV_PATH = rf"C:\Users\JTRUJILLO\Desktop\utiles\Reportes\historical_data\src\data\prices\{COUNTRY}-prices.csv"
# df_inventory = pd.read_csv(CSV_PATH)

Lectura desde archivo en google sheets

In [33]:
google_sheet_info = {
    "sheet_name": '[MKP] Precios',
    "worksheet": f'price_data_mx',
}
df_inventory = gsheets.read_sheet(google_sheet_info)

In [34]:
df_inventory[df_inventory["model"] == "RC250"]

,date,code,brand,model,year,status,price_base,discount_amount,price_net
215,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2025,available,31999.0,0.0,31999.0
433,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2023,discontinued,33999.0,0.0,33999.0
635,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2026,no_stock,31999.0,0.0,31999.0
1051,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2024,discontinued,31999.0,0.0,31999.0


## 3. Cargar inventario

In [35]:
df_inventory = df_inventory[df_inventory["brand"] == BRAND_NAME]
df_inventory = df_inventory[df_inventory["status"].isin(["available", "no_stock"])] # Modelos disponibles en marketplace

## 4. Definir URLs a scrapear (input manual)

In [36]:
if not urls_to_scrape:
    raise ValueError("Debes cargar al menos una URL en 'urls_to_scrape'")

print(f"URLs recibidas para scraping: {len(urls_to_scrape)}")

URLs recibidas para scraping: 24


## 5. Ejecutar price tracking

In [37]:
rows = run_price_tracking(
    urls=urls_to_scrape,
    brand_name=BRAND_NAME,
)

In [38]:
# Diagnostico (ejecutar una vez para verificar que change_status llega correctamente)
# import os, pprint
# from firecrawl import Firecrawl
# _fc = Firecrawl(api_key=os.getenv('FIRECRAWL_API_KEY'))
# _result = _fc.scrape(urls_to_scrape[0], formats=['markdown', {'type': 'changeTracking'}], only_main_content=True)
# _ct = getattr(_result, 'change_tracking', None)
# pprint.pprint(_ct)

In [39]:
df_scraped = pd.DataFrame(rows)
df_scraped

,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility
0,Bajaj,Pulsar LS 125,2025,https://www.motosbajaj.com.mx/product-page/pulsar-ls-125,"29,999.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T16:35:45.934+00:00,visible
1,Bajaj,Pulsar N 160 Carburada 2026,2026,https://www.motosbajaj.com.mx/product-page/n-160-carburada-2026,"38,599.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T16:35:46.855+00:00,visible
2,Bajaj,copia de pulsar n 160 fi abs ug,2026,https://www.motosbajaj.com.mx/product-page/copia-de-pulsar-n-160-fi-abs-ug,,oferta,MXN,2026-04-15T19:22:42.220395+00:00,removed,2026-04-15T16:02:09.697+00:00,visible
3,Bajaj,Boxer BM 150,2026,https://www.motosbajaj.com.mx/product-page/boxer-bm-150,"31,499.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T16:02:10.673+00:00,visible
4,Bajaj,dominar 400,2026,https://www.motosbajaj.com.mx/product-page/dominar-400,,oferta,MXN,2026-04-15T19:22:42.220395+00:00,removed,2026-04-15T16:02:09.698+00:00,visible
5,Bajaj,Pulsar NS 160,2026,https://www.motosbajaj.com.mx/product-page/pulsar-ns-160-ug,"48,499.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T16:02:17.271+00:00,visible
6,Bajaj,Pulsar NS 400 Z 2026,2026,https://www.motosbajaj.com.mx/product-page/pulsar-ns-400-z,"88,999.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T19:20:53.076+00:00,visible
7,Bajaj,Pulsar N 125 FI CBS,2026,https://www.motosbajaj.com.mx/product-page/pulsar-n-125-fi-cbs,"36,499.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T19:20:53.024+00:00,visible
8,Bajaj,Dominar 400 TE 2026,2026,https://www.motosbajaj.com.mx/product-page/dominar-400-touring-2026,"95,999.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T19:20:53.194+00:00,visible
9,Bajaj,Pulsar NS 200 2026,2026,https://www.motosbajaj.com.mx/product-page/pulsar-ns-200-ug-2026,"60,999.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T16:02:18.219+00:00,visible


## 6. Limpiar y mapear modelos

In [88]:
df_scraped["year_scraped"] = pd.to_numeric(df_scraped.get("year_scraped"), errors="coerce").astype("Int64")

df_scraped["model_name_clean"] = df_scraped["model_name"].apply(
    lambda model_name: clean_model_name(model_name, brand_name=BRAND_NAME)
)

mapeo = load_mapping_file(COUNTRY, BRAND_NAME)
df_scraped["model_mapped"] = df_scraped["model_name_clean"].apply(lambda x: map_model_name(x, mapeo))
df_scraped["model_mapped"] = df_scraped["model_mapped"].str.lower()

Archivo de mapeo cargado correctamente: C:\Users\JTRUJILLO\Desktop\utiles\Proyectos\tracking_prices_changes\src/data/json/replace_name/MX/bajaj_mapeo_nombres.json


## 7. Merge con inventario (inspeccion)

In [90]:
# 1) Preparar inventario (marketplace)
inv_for_merge = df_inventory.copy()
inv_for_merge["model_lower"] = inv_for_merge["model"].str.lower()
inv_for_merge["year"] = pd.to_numeric(inv_for_merge["year"], errors="coerce")

# 2) Preparar scrapeado
scraped_for_merge = df_scraped.copy()
scraped_for_merge["year_scraped"] = pd.to_numeric(
    scraped_for_merge.get("year_scraped"),
    errors="coerce",
)

# 3) Merge por modelo + año
# Comentario en español: Outer conserva tanto marketplace sin match como scrapeado sin match.
df_merged = pd.merge(
    inv_for_merge,
    scraped_for_merge,
    left_on=["model_lower", "year"],
    right_on=["model_mapped", "year_scraped"],
    how="outer",
    indicator=True,
)

df_merged.head(2)

,date,code,brand,model,year,status,price_base,discount_amount,price_net,model_lower,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility,model_name_clean,model_mapped,_merge
0,14/04/2026,MX1335-bajaj-avenger-220-cruise,Bajaj,Avenger 220 Cruise,2025.0,available,54499.0,3000.0,51499.0,avenger 220 cruise,Bajaj,Avenger 220 Cruise,2025,https://www.motosbajaj.com.mx/product-page/avenger-220,"51,499.00",oferta,MXN,2026-04-15T19:22:42.220395+00:00,same,2026-04-15T16:02:33.537+00:00,visible,Avenger 220 Cruise,avenger 220 cruise,both
1,14/04/2026,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,2024.0,no_stock,30999.0,0.0,30999.0,boxer bm 150,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


## 8. Construir comparacion de precios

In [91]:
df_final = build_price_comparison(
    df_scraped,
    df_inventory,
    COUNTRY,
    galgo_fee=BRAND_FEES.get(BRAND_NAME.lower(), 0),
)

## 9. Inspeccion

In [ ]:
# Diagnostico rapido de cobertura del merge
print(df_merged["_merge"].value_counts(dropna=False))

solo_marketplace = df_merged[df_merged["_merge"] == "left_only"][["brand", "model", "year"]].drop_duplicates()
solo_scrapeado = df_merged[df_merged["_merge"] == "right_only"][["model_name", "model_mapped", "year_scraped", "url"]].drop_duplicates()
# print(f"Solo marketplace: {len(solo_marketplace)}")
# print(f"Solo scrapeado: {len(solo_scrapeado)}")
# solo_scrapeado.head(20)

In [96]:
from datetime import date

today_str = date.today().strftime("%d%m%Y")
print(today_str)

15042026


## 10. Exportar

In [97]:
# https://docs.google.com/spreadsheets/d/1UuwjLSfpOF-fYhOSv7Tw_iIJeqaAkItDZWURfo4cy1Q/edit?gid=122865928#gid=122865928

In [98]:
output_info = {
    "sheet_name": "[Scraping - MX] Comparativa de precios",
    "worksheet": BRAND_NAME,  # "Italika" o "Bajaj"
    "df": df_final.sort_values(by="model_name"),
}

gsheets.update_sheet(output_info, clear_data=True)
print(f"Datos escritos en '{output_info['sheet_name']}' > hoja '{output_info['worksheet']}' — {len(df_final)} filas")

Updated sheet: [Scraping - MX] Comparativa de precios
Datos escritos en '[Scraping - MX] Comparativa de precios' > hoja 'Bajaj' — 28 filas


## 11. Registrar diferencias en log histórico

In [99]:
from src.utils.price_diff_log import append_price_diff_log

LOG_PATH = "../data/logs/price_diff_log.csv"

rows_logged = append_price_diff_log(df_final, LOG_PATH)

if rows_logged:
    print(f"{rows_logged} modelos con diferencia de precio registrados en el log.")
else:
    print("Sin diferencias de precio en esta ejecución — nada agregado al log.")

8 modelos con diferencia de precio registrados en el log.


---
## Anexo: correr sobre catalogo completo

In [ ]:
# # Descomentar para correr con una lista mas grande de URLs
# rows_full = run_price_tracking(
#     urls=urls_to_scrape,
#     brand_name=BRAND_NAME,
# )
# df_full = pd.DataFrame(rows_full)
# print(f"Modelos procesados: {len(df_full)}")
# print(df_full["change_status"].value_counts().to_dict())
# df_full.head()

: 